# Copywallet Data Analysis

Query the bot's SQLite database to understand what's happening:
- Which wallets are performing best?
- Which filters reject the most trades?
- How are paper trades performing?
- Is the Claude Brain finding edge?

**Run this anytime to see the current state. The bot writes to `copywallet.db` continuously.**

In [ ]:
import sqlite3
import pandas as pd
import json
from datetime import datetime

db = sqlite3.connect('copywallet.db')
db.row_factory = sqlite3.Row

def query(sql):
    return pd.read_sql_query(sql, db)

print(f"Database loaded at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Tables: {[r[0] for r in db.execute(\"SELECT name FROM sqlite_master WHERE type='table'\").fetchall()]}")

## 1. Tracked Wallets Overview

In [ ]:
wallets = query("""
    SELECT 
        username,
        address,
        total_pnl,
        total_volume,
        is_suspicious,
        category_stats,
        last_scored
    FROM wallet_profiles 
    ORDER BY total_pnl DESC
""")

print(f"Total wallets stored: {len(wallets)}")
print(f"Non-suspicious (copy-eligible): {len(wallets[wallets['is_suspicious'] == 0])}")
print(f"Suspicious (data only): {len(wallets[wallets['is_suspicious'] == 1])}")
print()

# Show non-suspicious wallets with their category stats
for _, w in wallets[wallets['is_suspicious'] == 0].iterrows():
    stats = json.loads(w['category_stats']) if w['category_stats'] else {}
    cats = []
    for cat, s in stats.items():
        wr = s.get('win_rate', 0)
        pos = s.get('resolved_positions', 0)
        cats.append(f"{cat}: {wr:.0%} WR ({pos} trades)")
    print(f"{w['username']:25s} PnL=${w['total_pnl']:>10,.0f}  |  {', '.join(cats)}")

## 2. Paper Trades

In [ ]:
trades = query("""
    SELECT 
        timestamp,
        question,
        category,
        side,
        price,
        size_usd,
        fee,
        signal_source,
        status,
        leader_address
    FROM trades 
    ORDER BY timestamp DESC
""")

if len(trades) == 0:
    print("No trades yet. The bot is waiting for leaders to trade active markets.")
else:
    print(f"Total trades: {len(trades)}")
    print(f"By source: {trades['signal_source'].value_counts().to_dict()}")
    print(f"By status: {trades['status'].value_counts().to_dict()}")
    print(f"By category: {trades['category'].value_counts().to_dict()}")
    print(f"\nTotal size: ${trades['size_usd'].sum():,.2f}")
    print(f"Total fees: ${trades['fee'].sum():,.2f}")
    print()
    trades[['timestamp', 'question', 'side', 'price', 'size_usd', 'signal_source', 'status']].head(20)

## 3. Open Positions

In [ ]:
positions = query("""
    SELECT 
        market_id,
        side,
        entry_price,
        size_shares,
        size_usd,
        current_price,
        unrealized_pnl,
        signal_source,
        leader_address,
        opened_at
    FROM positions
""")

if len(positions) == 0:
    print("No open positions.")
else:
    print(f"Open positions: {len(positions)}")
    print(f"Total exposure: ${positions['size_usd'].sum():,.2f}")
    print(f"Unrealized PnL: ${positions['unrealized_pnl'].sum():,.2f}")
    positions

## 4. Portfolio History

In [ ]:
portfolio = query("""
    SELECT timestamp, total_value, cash_balance, positions_value, daily_pnl, total_pnl
    FROM portfolio 
    ORDER BY timestamp ASC
""")

if len(portfolio) == 0:
    print("No portfolio snapshots yet. Wait for the first 5-minute snapshot cycle.")
else:
    print(f"Snapshots: {len(portfolio)}")
    print(f"Latest value: ${portfolio.iloc[-1]['total_value']:,.2f}")
    print(f"Total PnL: ${portfolio.iloc[-1]['total_pnl']:,.2f}")
    print()
    
    try:
        import matplotlib.pyplot as plt
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
        
        portfolio['ts'] = pd.to_datetime(portfolio['timestamp'])
        
        ax1.plot(portfolio['ts'], portfolio['total_value'], color='#10B981', linewidth=2)
        ax1.set_title('Portfolio Value Over Time')
        ax1.set_ylabel('USD')
        ax1.grid(True, alpha=0.3)
        
        ax2.bar(portfolio['ts'], portfolio['total_pnl'], 
                color=['#10B981' if x >= 0 else '#EF4444' for x in portfolio['total_pnl']])
        ax2.set_title('Cumulative PnL')
        ax2.set_ylabel('USD')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("Install matplotlib for charts: pip install matplotlib")

## 5. Known Trades (Activity Monitoring)

In [ ]:
known = query("""
    SELECT 
        wallet_address,
        COUNT(*) as trade_count,
        MIN(detected_at) as first_seen,
        MAX(detected_at) as last_seen
    FROM known_trades
    GROUP BY wallet_address
    ORDER BY trade_count DESC
""")

if len(known) == 0:
    print("No trades detected yet.")
else:
    print(f"Wallets with detected activity: {len(known)}")
    print(f"Total trades observed: {known['trade_count'].sum()}")
    print()
    
    # Join with wallet names
    for _, k in known.iterrows():
        name_row = db.execute(
            "SELECT username FROM wallet_profiles WHERE address = ?", 
            (k['wallet_address'],)
        ).fetchone()
        name = name_row[0] if name_row else k['wallet_address'][:12]
        print(f"{name:25s} {k['trade_count']:>5} trades  |  {k['first_seen'][:16]} → {k['last_seen'][:16]}")

## 6. Market Cache (Markets Seen)

In [ ]:
markets = query("""
    SELECT category, COUNT(*) as count
    FROM market_cache
    GROUP BY category
    ORDER BY count DESC
""")

if len(markets) == 0:
    print("No markets cached yet.")
else:
    print(f"Total markets seen: {markets['count'].sum()}")
    print()
    for _, m in markets.iterrows():
        print(f"  {m['category']:20s} {m['count']:>5} markets")

## 7. Wallet Performance Comparison

Compare all tracked wallets side by side — useful for deciding who to promote to Tier 3 (copy list) later.

In [ ]:
all_wallets = query("SELECT * FROM wallet_profiles ORDER BY total_pnl DESC")

rows = []
for _, w in all_wallets.iterrows():
    stats = json.loads(w['category_stats']) if w['category_stats'] else {}
    for cat, s in stats.items():
        rows.append({
            'wallet': w['username'] or w['address'][:12],
            'category': cat,
            'win_rate': s.get('win_rate', 0),
            'positions': s.get('resolved_positions', 0),
            'profit_factor': s.get('profit_factor', 0),
            'avg_size': s.get('avg_position_size', 0),
            'total_pnl': s.get('total_pnl', 0),
            'suspicious': bool(w['is_suspicious']),
        })

if rows:
    df = pd.DataFrame(rows)
    
    # Non-suspicious only
    legit = df[~df['suspicious']].sort_values('total_pnl', ascending=False)
    print("=== COPY-ELIGIBLE WALLETS (by category) ===")
    print(legit[['wallet', 'category', 'win_rate', 'positions', 'profit_factor', 'total_pnl']].to_string(index=False))
    
    print(f"\n=== SUSPICIOUS WALLETS (data only, {len(df[df['suspicious']])} entries) ===")
    sus = df[df['suspicious']].sort_values('total_pnl', ascending=False).head(10)
    print(sus[['wallet', 'category', 'win_rate', 'positions', 'profit_factor']].to_string(index=False))
else:
    print("No wallet data yet.")

## 8. Graduation Readiness

Check if the bot meets the criteria to switch from paper to live trading.

In [ ]:
trades_df = query("SELECT * FROM trades WHERE status = 'filled'")
portfolio_df = query("SELECT * FROM portfolio ORDER BY timestamp ASC")

total_trades = len(trades_df)

if total_trades == 0:
    print("No filled trades yet — graduation criteria not applicable.")
else:
    # Calculate metrics
    first_trade = trades_df['timestamp'].min()
    days_trading = (datetime.now() - datetime.fromisoformat(first_trade)).days
    
    # Win rate (simplified — based on PnL of closed positions)
    # For a proper win rate we'd need resolution data
    
    criteria = {
        'Trades >= 200': ('PASS' if total_trades >= 200 else 'FAIL', f'{total_trades}/200'),
        'Days >= 14': ('PASS' if days_trading >= 14 else 'FAIL', f'{days_trading}/14'),
    }
    
    if len(portfolio_df) > 0:
        max_drawdown = 0
        peak = portfolio_df['total_value'].iloc[0]
        for val in portfolio_df['total_value']:
            peak = max(peak, val)
            dd = (peak - val) / peak
            max_drawdown = max(max_drawdown, dd)
        criteria['Drawdown < 15%'] = ('PASS' if max_drawdown < 0.15 else 'FAIL', f'{max_drawdown:.1%}')
    
    print("=== GRADUATION CHECKLIST ===")
    all_pass = True
    for name, (status, value) in criteria.items():
        icon = '✅' if status == 'PASS' else '❌'
        print(f"  {icon} {name}: {value}")
        if status == 'FAIL':
            all_pass = False
    
    print()
    if all_pass:
        print("🟢 ALL CRITERIA MET — ready to go live!")
    else:
        print("🔴 Not ready yet. Keep paper trading.")

In [ ]:
db.close()
print("Database closed. Re-run cells to refresh data.")